In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q18/annotated/checkpoints/pre_cell_3.pickle

trying: ['lineitem']


me:  1
trying: ['load_orders']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['load_customer']
me:  3
trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0
trying: ['customer']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['orders']


me:  2


Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# build a mapping from C_CUSTKEY to C_NAME on the GPU
cust_map = customer.set_index("C_CUSTKEY")["C_NAME"]

# 1) group and sum L_QUANTITY by order, 2) filter >300
gb1 = (
    lineitem
      .groupby("L_ORDERKEY", as_index=False, sort=False)["L_QUANTITY"]
      .sum()
)
gb1 = gb1[gb1.L_QUANTITY > 300]

# join with the needed orders columns
df = gb1.merge(
    orders[["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_TOTALPRICE"]],
    left_on="L_ORDERKEY", right_on="O_ORDERKEY"
)

# map in the customer name instead of a second merge
df["C_NAME"] = df["O_CUSTKEY"].map(cust_map)

# rename, select and sort
df = df.rename(columns={"O_CUSTKEY": "C_CUSTKEY"})
total = (
    df[["C_NAME", "C_CUSTKEY", "O_ORDERKEY", "O_ORDERDATE", "O_TOTALPRICE", "L_QUANTITY"]]
      .sort_values(["O_TOTALPRICE", "O_ORDERDATE"], ascending=[False, True])
)

CPU times: user 74.3 ms, sys: 51.5 ms, total: 126 ms
Wall time: 128 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q18/rewritten/o4_mini_high/checkpoints/post_cell_3_try_1.pickle

migration speed (bps): 810884317.8748213
---------------------------
variables to migrate:
customer 48
pd 72
orders 48
load_orders 144
load_customer 144
gb1 48
total 48
tpch_parent 54
DATA_ROOT 80
df 48
lineitem 48
cust_map 48
load_lineitem 144
STORAGE_OPTS 64
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q18/rewritten/o4_mini_high/checkpoints/post_cell_3_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['orders', 'lineitem']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['total', 'df', 'cust_map', 'gb1']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q18/opt_cell_exec_info_3_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q18/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['jn2']
me:  4
trying: ['jn1']
me:  4
trying: ['lineitem']


me:  1
trying: ['customer']
me:  3
trying: ['load_lineitem']
me:  1
trying: ['pd']
me:  0
trying: ['fgb1']
me:  4
trying: ['gb1']
me:  4
trying: ['STORAGE_OPTS']
me:  0
trying: ['orders']


me:  2
trying: ['DATA_ROOT']
me:  0
trying: ['load_orders']
me:  2
trying: ['load_customer']
me:  3
trying: ['total']
me:  4
trying: ['gb2']
me:  4
trying: ['orig_output']
me:  5
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable orders
Declaring variable load_orders
Declaring variable customer
Declaring variable load_customer
Declaring variable jn2
Declaring variable jn1
Declaring variable fgb1
Declaring variable gb1
Declaring variable total
Declaring variable gb2
Declaring variable orig_output
